# LLM-as-a-Judge: The Foundation of Modern LLM Evaluation

This notebook is **the prerequisite** to every other workbook in this series. Before you learn DeepEval, RAGAS, or any evaluation framework, you need to understand the core engine they all run on: **using a powerful LLM to judge the output of another LLM**.

By the end of this notebook you will:

1. Understand why traditional metrics (BLEU, ROUGE, exact match) fail for LLM evaluation
2. Build an LLM-as-a-Judge system from scratch using the OpenAI API
3. Implement the three judge paradigms: pointwise, pairwise, and reference-based
4. Discover and demonstrate four major judge biases
5. See exactly how DeepEval and RAGAS implement LLM-as-a-Judge under the hood
6. Customize the judge: different models, rubrics, temperatures
7. Know when (and when NOT) to use LLM-as-a-Judge

-

## Part 1: What is LLM-as-a-Judge?

### The Problem with Traditional Metrics

For decades, NLP researchers used metrics like:

| Metric | How it works | Why it fails for LLMs |
|---|---|---|
| **BLEU** | N-gram overlap with reference | "Paris is the capital of France" vs "France's capital is Paris" get a low score despite being semantically identical |
| **ROUGE** | Recall-oriented n-gram overlap | Same problem — penalizes paraphrasing |
| **Exact Match** | String equality | Absurdly strict — any rephrasing is a failure |
| **F1 (token)** | Token-level precision/recall | Better, but still ignores meaning |

These metrics compare **strings**, not **meaning**. They were designed for machine translation and summarization where there is a fixed reference answer. But LLM outputs are creative, varied, and context-dependent. A response can be completely correct while sharing zero n-grams with the reference.

### The Core Idea: A Smart Evaluator

What if, instead of counting matching words, we asked a **smart evaluator** that actually understands language?

That is exactly what LLM-as-a-Judge does: use a powerful LLM (typically GPT-4 or GPT-4o) to evaluate the output of another LLM (or the same LLM). The judge reads the question, the response, optionally a reference answer or context, and produces a score with a justification.

**Analogy:** Think of it like having a senior expert review a junior employee's work. The junior writes a report (the LLM response), and the senior (the judge LLM) reads it and gives feedback: "This is 4 out of 5 because the main points are correct but the conclusion is vague."

### The Three Paradigms

LLM-as-a-Judge comes in three flavors:

1. **Pointwise Scoring**: Rate a single response on a scale (e.g., 1-5). "How good is this response?"
2. **Pairwise Comparison**: Given two responses, which one is better? "Is A or B the better answer?"
3. **Reference-Based**: Compare a response to a gold-standard reference. "How close is this to the ideal answer?"

We will build all three from scratch in Part 2.

-

## Part 2: Building LLM-as-a-Judge from Scratch

### Setup

First, let's load our environment and set up the OpenAI client.

In [ ]:
import os
import json
import textwrap
from dotenv import load_dotenv

load_dotenv("../.env")

from openai import OpenAI

client = OpenAI()  # picks up OPENAI_API_KEY from environment

# Quick sanity check
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'ready' and nothing else."}],
    max_tokens=10,
)
print("OpenAI client ready:", resp.choices[0].message.content)

### 2.1 The Simplest Judge: Naive Prompting

The most naive approach: just ask the LLM to rate a response on a 1-5 scale. No structure, no rubric. Let's see what happens.

In [ ]:
query = "What causes seasons on Earth?"
response_text = (
    "Seasons are caused by the tilt of Earth's axis (about 23.5 degrees). "
    "As Earth orbits the Sun, different hemispheres receive varying amounts "
    "of direct sunlight, leading to warmer summers and cooler winters."
)

naive_prompt = f"""Rate the following response on a scale of 1 to 5.

Question: {query}
Response: {response_text}

Score:"""

result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": naive_prompt}],
    temperature=0,
    max_tokens=200,
)

print("Raw judge output:")
print(result.choices[0].message.content)

**Problems with this approach:**

- The output format is unpredictable: sometimes it returns just "5", sometimes "5/5", sometimes a paragraph
- There is no rubric: what does "3" mean? The LLM decides on its own
- No reasoning: we cannot understand *why* it gave that score
- Difficult to parse programmatically

Let's fix all of these problems.

### 2.2 Better Judge with Structured Output and Rubric

A good LLM judge needs:

1. **A clear rubric** defining what each score means
2. **Structured output** (JSON) so we can parse the result reliably
3. **Mandatory reasoning** so the score is explainable

Let's define a generic quality rubric and build a proper judge prompt.

In [ ]:
RUBRIC = """
Score 1 (Terrible): The response is completely wrong, irrelevant, or incoherent.
Score 2 (Poor): The response addresses the topic but contains major errors or is mostly unhelpful.
Score 3 (Adequate): The response is partially correct and somewhat helpful, but has notable gaps or minor errors.
Score 4 (Good): The response is mostly correct, relevant, and helpful with only minor issues.
Score 5 (Excellent): The response is fully correct, comprehensive, clear, and directly addresses the question.
"""

JUDGE_SYSTEM_PROMPT = f"""You are an impartial judge evaluating the quality of an AI assistant's response to a user question.

Use the following rubric:
{RUBRIC}

You MUST respond with valid JSON in this exact format:
{{
  "score": <integer 1-5>,
  "reason": "<one or two sentence justification>"
}}

Do not include any other text outside the JSON."""

print("Judge system prompt defined.")
print("Rubric covers scores 1 (Terrible) through 5 (Excellent).")

In [ ]:
def structured_judge(query: str, response_text: str, model: str = "gpt-4o-mini") -> dict:
    """Judge a response using a structured rubric. Returns {score, reason}."""
    user_prompt = f"""Evaluate the following response.

Question: {query}
Response: {response_text}"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )

    raw = result.choices[0].message.content.strip()
    # Parse JSON from the response
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        # Try to extract JSON from markdown code blocks
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
        else:
            parsed = {"score": -1, "reason": f"Failed to parse: {raw}"}
    return parsed

print("structured_judge() function defined.")

Now let's test the structured judge on several responses of varying quality to see if the rubric produces meaningful, differentiated scores.

In [ ]:
query = "What causes seasons on Earth?"

test_responses = {
    "excellent": (
        "Seasons are caused by the tilt of Earth's rotational axis (approximately 23.5 degrees) "
        "relative to its orbital plane around the Sun. As Earth orbits the Sun over the course of "
        "a year, the tilt causes different hemispheres to receive varying amounts of direct sunlight. "
        "When the Northern Hemisphere is tilted toward the Sun, it experiences summer, while the "
        "Southern Hemisphere experiences winter, and vice versa."
    ),
    "adequate": (
        "The seasons happen because of how the Earth moves around the Sun. "
        "Sometimes we are closer and sometimes farther away."
    ),
    "poor": (
        "Seasons are caused by the Sun getting hotter and cooler throughout the year."
    ),
    "irrelevant": (
        "The stock market has been volatile this quarter due to inflation concerns. "
        "Investors should diversify their portfolios."
    ),
}

print(f"Question: {query}\n")
print("=" * 70)

for label, resp in test_responses.items():
    judgment = structured_judge(query, resp)
    print(f"\n[{label.upper()}] Score: {judgment['score']}/5")
    print(f"  Response: {resp[:80]}...")
    print(f"  Reason:   {judgment['reason']}")

You should see a clear gradient: the excellent response scores 5, the adequate response around 2-3 (it contains the common misconception about distance rather than tilt), the poor response gets 1-2, and the irrelevant response gets 1. This demonstrates that even a simple structured judge with a rubric can differentiate quality levels.

-

### 2.3 Pointwise Scoring with Criteria

A pointwise judge rates a single response along a specific **criterion** (e.g., relevancy, helpfulness, conciseness). Different criteria can yield very different scores for the same response.

Let's build a flexible pointwise judge that accepts any evaluation criterion.

In [ ]:
def pointwise_judge(query: str, response_text: str, criteria: str, model: str = "gpt-4o-mini") -> dict:
    """
    Rate a response on a 1-5 scale for a specific criterion.
    
    Args:
        query: The user's question
        response_text: The LLM's response
        criteria: The evaluation criterion (e.g., 'relevancy', 'helpfulness')
        model: The judge model to use
    
    Returns:
        dict with 'score' (1-5) and 'reason'
    """
    system_prompt = f"""You are an impartial judge. Evaluate the response ONLY on the criterion: {criteria}.

Scoring rubric for {criteria}:
  1 = Completely fails this criterion
  2 = Mostly fails, with minor redeeming aspects
  3 = Partially meets this criterion
  4 = Mostly meets this criterion with minor shortcomings
  5 = Fully meets this criterion

Respond with valid JSON only:
{{"score": <1-5>, "reason": "<brief justification>"}}"""

    user_prompt = f"""Question: {query}
Response: {response_text}

Evaluate on: {criteria}"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"score": -1, "reason": f"Parse error: {raw}"}

print("pointwise_judge() function defined.")

Now let's demonstrate how the **same response** gets different scores depending on which criterion we evaluate. We will use a verbose but accurate response and score it on three criteria: relevancy, helpfulness, and conciseness.

In [ ]:
query = "What is the boiling point of water?"

verbose_response = (
    "Great question! Water is a fascinating molecule composed of two hydrogen atoms and one oxygen atom (H2O). "
    "It exists in three states: solid, liquid, and gas. The boiling point of water at standard atmospheric "
    "pressure (1 atm or 101.325 kPa) is 100 degrees Celsius (212 degrees Fahrenheit). This is the temperature "
    "at which water transitions from a liquid to a gaseous state. At higher altitudes, where atmospheric pressure "
    "is lower, water boils at a lower temperature. For example, at the top of Mount Everest, water boils at "
    "approximately 70 degrees Celsius. The boiling point is also affected by dissolved substances; adding salt "
    "to water raises its boiling point slightly, a phenomenon known as boiling point elevation. This principle "
    "is widely used in cooking and industrial processes."
)

criteria_list = ["relevancy", "helpfulness", "conciseness"]

print(f"Question: {query}")
print(f"Response: {verbose_response[:100]}...\n")
print("=" * 60)

for criterion in criteria_list:
    result = pointwise_judge(query, verbose_response, criterion)
    print(f"\n  {criterion.upper()}: {result['score']}/5")
    print(f"  Reason: {result['reason']}")

Notice how the same response likely scores high on relevancy and helpfulness (it correctly answers the question and provides useful context) but lower on conciseness (it includes a lot of extra information). This is the power of criterion-specific evaluation: it lets you measure exactly the dimension of quality you care about.

-

### 2.4 Pairwise Comparison

Instead of scoring a single response, a pairwise judge compares two responses and decides which is better. This is often more reliable than pointwise scoring because relative comparisons are easier for both humans and LLMs than absolute ratings.

However, pairwise comparison has a well-known flaw: **position bias** (the judge tends to prefer whichever response is listed first). We will demonstrate this and implement a debiasing technique.

In [ ]:
def pairwise_judge(query: str, response_a: str, response_b: str, model: str = "gpt-4o-mini") -> dict:
    """
    Compare two responses and determine which is better.
    
    Returns:
        dict with 'winner' ('A' or 'B' or 'tie') and 'reason'
    """
    system_prompt = """You are an impartial judge comparing two AI responses to a user question.
Evaluate both responses on overall quality: correctness, relevance, helpfulness, and clarity.

Respond with valid JSON only:
{"winner": "A" or "B" or "tie", "reason": "<brief justification>"}"""

    user_prompt = f"""Question: {query}

Response A: {response_a}

Response B: {response_b}

Which response is better?"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"winner": "error", "reason": f"Parse error: {raw}"}

print("pairwise_judge() function defined.")

Let's test with a clear winner case first.

In [ ]:
query = "What is the capital of Japan?"

good_response = "The capital of Japan is Tokyo. It has been the capital since 1868 when the Emperor moved there from Kyoto."
bad_response = "The capital of Japan is Osaka. It is known for its street food."

result = pairwise_judge(query, good_response, bad_response)
print(f"Winner: Response {result['winner']}")
print(f"Reason: {result['reason']}")

Now let's demonstrate **position bias**. We will use two responses of similar quality and swap their positions to see if the judge's preference changes.

In [ ]:
query = "Explain the concept of supply and demand in economics."

response_1 = (
    "Supply and demand is a fundamental economic model. Demand refers to how much of a product "
    "consumers want to buy at various prices - generally, lower prices lead to higher demand. "
    "Supply refers to how much producers are willing to sell at various prices - higher prices "
    "incentivize more supply. The equilibrium price is where supply meets demand."
)

response_2 = (
    "In economics, supply and demand describes the relationship between sellers and buyers in a market. "
    "The demand curve shows that people buy more when prices are low. The supply curve shows that "
    "producers make more when prices are high. Where these curves intersect is the market equilibrium, "
    "which determines the price and quantity of goods traded."
)

# Test 1: response_1 as A, response_2 as B
result_ab = pairwise_judge(query, response_1, response_2)
print("Order 1: response_1=A, response_2=B")
print(f"  Winner: {result_ab['winner']}")
print(f"  Reason: {result_ab['reason']}")

# Test 2: Swap positions
result_ba = pairwise_judge(query, response_2, response_1)
print(f"\nOrder 2: response_2=A, response_1=B")
print(f"  Winner: {result_ba['winner']}")
print(f"  Reason: {result_ba['reason']}")

# Analysis
print("\n" + "=" * 60)
if result_ab['winner'] == 'A' and result_ba['winner'] == 'A':
    print("POSITION BIAS DETECTED: The judge always preferred whichever response was listed as 'A'.")
elif result_ab['winner'] == result_ba['winner']:
    print(f"Consistent: The judge preferred the same response in both orderings.")
else:
    winner_1 = "response_1" if result_ab['winner'] == 'A' else "response_2"
    winner_2 = "response_2" if result_ba['winner'] == 'A' else "response_1"
    print(f"Order 1 preferred: {winner_1}")
    print(f"Order 2 preferred: {winner_2}")
    if winner_1 == winner_2:
        print("Consistent result despite position swap.")
    else:
        print("INCONSISTENCY detected - results changed when positions were swapped.")

### Debiasing Pairwise Comparison

The standard mitigation for position bias is to run the comparison **twice** (swapping the order) and take the consensus. If the two runs disagree, it is declared a tie. This is exactly what many evaluation frameworks (including Chatbot Arena) do.

In [ ]:
def debiased_pairwise_judge(query: str, response_a: str, response_b: str, model: str = "gpt-4o-mini") -> dict:
    """
    Run pairwise comparison in both orderings and take consensus.
    If the two orderings disagree, declare a tie.
    """
    # Run 1: A first, B second
    result_1 = pairwise_judge(query, response_a, response_b, model)
    # Map the winner to the original labels
    winner_1 = result_1['winner']  # 'A', 'B', or 'tie'
    
    # Run 2: B first, A second (swapped)
    result_2 = pairwise_judge(query, response_b, response_a, model)
    # Map back: if winner is 'A' in run 2, that means response_b won (since B was listed as A)
    winner_2_mapped = {'A': 'B', 'B': 'A', 'tie': 'tie'}.get(result_2['winner'], 'tie')
    
    # Consensus
    if winner_1 == winner_2_mapped:
        final_winner = winner_1
        consensus = "agree"
    else:
        final_winner = "tie"
        consensus = "disagree"
    
    return {
        "winner": final_winner,
        "consensus": consensus,
        "run1": {"order": "A=original_A, B=original_B", "winner": winner_1, "reason": result_1['reason']},
        "run2": {"order": "A=original_B, B=original_A", "winner": result_2['winner'], "mapped_winner": winner_2_mapped, "reason": result_2['reason']},
    }

# Test debiased comparison
result = debiased_pairwise_judge(query, response_1, response_2)

print("Debiased Pairwise Comparison")
print("=" * 60)
print(f"Final Winner: {result['winner']} (consensus: {result['consensus']})")
print(f"\nRun 1 ({result['run1']['order']}):")
print(f"  Winner: {result['run1']['winner']} - {result['run1']['reason']}")
print(f"\nRun 2 ({result['run2']['order']}):")
print(f"  Raw winner: {result['run2']['winner']}, Mapped: {result['run2']['mapped_winner']}")
print(f"  Reason: {result['run2']['reason']}")

-

### 2.5 Reference-Based Scoring

The third paradigm compares a candidate response against a **gold-standard reference answer**. This is useful when you have human-written ideal answers. The judge evaluates how closely the candidate captures the same information as the reference.

In [ ]:
def reference_judge(query: str, response_text: str, reference: str, model: str = "gpt-4o-mini") -> dict:
    """
    Score how well a response matches a reference answer.
    
    Returns:
        dict with 'score' (1-5) and 'reason'
    """
    system_prompt = """You are an impartial judge. Compare the candidate response to the reference answer.

Scoring rubric:
  1 = Completely different or contradicts the reference
  2 = Captures very little of the reference information
  3 = Captures some key points but misses important details
  4 = Captures most key points with minor omissions or additions
  5 = Captures all key points from the reference (paraphrasing is fine)

Respond with valid JSON only:
{"score": <1-5>, "reason": "<brief justification>"}"""

    user_prompt = f"""Question: {query}

Reference Answer: {reference}

Candidate Response: {response_text}"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"score": -1, "reason": f"Parse error: {raw}"}

print("reference_judge() function defined.")

Let's test the reference-based judge with four candidate responses of varying similarity to the reference.

In [ ]:
query = "What is photosynthesis?"
reference = (
    "Photosynthesis is the biological process by which green plants and certain other organisms "
    "convert light energy (usually from the Sun) into chemical energy stored in glucose. It occurs "
    "primarily in the chloroplasts of plant cells, using carbon dioxide from the air and water from "
    "the soil. Oxygen is released as a byproduct."
)

candidates = {
    "exact_paraphrase": (
        "Photosynthesis is a process where plants use sunlight to turn CO2 and water into glucose "
        "and oxygen. This happens in the chloroplasts of their cells."
    ),
    "partial_match": (
        "Photosynthesis is how plants make food using sunlight. It involves the green parts of the plant."
    ),
    "wrong_but_related": (
        "Photosynthesis is the process where plants absorb oxygen and release carbon dioxide "
        "to produce energy for growth."
    ),
    "completely_wrong": (
        "Photosynthesis is a type of nuclear fusion that occurs in the Earth's core, "
        "generating heat that sustains life on the surface."
    ),
}

print(f"Question: {query}")
print(f"Reference: {reference[:80]}...\n")
print("=" * 70)

for label, candidate in candidates.items():
    result = reference_judge(query, candidate, reference)
    print(f"\n[{label.upper()}] Score: {result['score']}/5")
    print(f"  Candidate: {candidate[:80]}...")
    print(f"  Reason:    {result['reason']}")

The reference-based judge should clearly separate the paraphrase (high score) from the wrong answer (low score), with the partial match and the reversed-facts response falling in between. This paradigm is especially useful for RAG evaluation where you have ground-truth answers.

-

## Part 3: Judge Biases -- Demonstrated with Code

LLM judges are not perfect. They exhibit systematic biases that can skew evaluation results. Understanding these biases is critical before relying on LLM-as-a-Judge in production.

We will demonstrate four major biases:

1. **Position Bias** (already shown above)
2. **Verbosity Bias**: longer responses score higher, regardless of quality
3. **Self-Enhancement Bias**: the judge rates its own outputs higher
4. **Format Bias**: well-formatted responses (markdown, JSON) score higher than plain text

### 3.1 Verbosity Bias

LLM judges tend to prefer verbose responses even when a concise response is equally or more accurate. Let's demonstrate this by creating a short, accurate response and a verbose, slightly less accurate response.

In [ ]:
query = "What is the speed of light?"

concise_response = "The speed of light in a vacuum is approximately 299,792,458 meters per second (about 3 x 10^8 m/s)."

verbose_response = (
    "That's a great question! The speed of light is one of the most fundamental constants in physics. "
    "Light travels at different speeds depending on the medium it passes through. In a vacuum, which is "
    "the absence of matter, light travels at its maximum speed. This speed was first measured with "
    "reasonable accuracy by Ole Roemer in 1676 using observations of Jupiter's moon Io. Over the centuries, "
    "scientists have refined this measurement using increasingly sophisticated equipment. Today, the speed "
    "of light in a vacuum is defined as exactly 299,792,458 meters per second. This value is used to "
    "define the meter itself in the International System of Units. Albert Einstein's theory of special "
    "relativity, published in 1905, showed that the speed of light is the cosmic speed limit - nothing "
    "with mass can reach or exceed this speed. The speed of light is commonly denoted by the letter 'c' "
    "in physics equations, most famously in E=mc^2."
)

# Score both
concise_score = structured_judge(query, concise_response)
verbose_score = structured_judge(query, verbose_response)

print("VERBOSITY BIAS DEMONSTRATION")
print("=" * 60)
print(f"\nConcise response ({len(concise_response)} chars): Score {concise_score['score']}/5")
print(f"  Reason: {concise_score['reason']}")
print(f"\nVerbose response ({len(verbose_response)} chars): Score {verbose_score['score']}/5")
print(f"  Reason: {verbose_score['reason']}")

if verbose_score['score'] > concise_score['score']:
    print("\n>>> VERBOSITY BIAS: The longer response scored higher despite both being correct.")
elif verbose_score['score'] == concise_score['score']:
    print("\n>>> No verbosity bias detected here (both scored equally).")
else:
    print("\n>>> Concise response actually scored higher -- no verbosity bias.")

### Mitigating Verbosity Bias

We can mitigate verbosity bias by explicitly instructing the judge that **brevity should not be penalized** and that **unnecessary verbosity should not be rewarded**.

In [ ]:
DEBIASED_JUDGE_PROMPT = """You are an impartial judge evaluating the quality of an AI response.

IMPORTANT: Do NOT favor longer responses. A concise, accurate response is just as good as
a verbose one. Evaluate based on correctness and relevance, NOT length. Unnecessary verbosity
should be considered a slight negative, not a positive.

Rubric:
  1 = Completely wrong or irrelevant
  2 = Mostly wrong with minor correct elements
  3 = Partially correct but has notable gaps or errors
  4 = Mostly correct and relevant with minor issues
  5 = Fully correct, relevant, and appropriately detailed

Respond with valid JSON: {"score": <1-5>, "reason": "<justification>"}"""

def debiased_judge(query: str, response_text: str, model: str = "gpt-4o-mini") -> dict:
    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": DEBIASED_JUDGE_PROMPT},
            {"role": "user", "content": f"Question: {query}\nResponse: {response_text}"},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        return json.loads(match.group()) if match else {"score": -1, "reason": raw}

# Re-test with debiased judge
concise_debiased = debiased_judge(query, concise_response)
verbose_debiased = debiased_judge(query, verbose_response)

print("DEBIASED JUDGE RESULTS")
print("=" * 60)
print(f"Concise: {concise_debiased['score']}/5 - {concise_debiased['reason']}")
print(f"Verbose: {verbose_debiased['score']}/5 - {verbose_debiased['reason']}")

### 3.4 Summary of Biases

| Bias | Description | Mitigation |
|---|---|---|
| **Position Bias** | First-listed response preferred in pairwise comparison | Run both orderings, take consensus |
| **Verbosity Bias** | Longer responses score higher | Add "do not favor length" to rubric |
| **Self-Enhancement** | Model rates its own outputs higher | Use a different/stronger model as judge |
| **Format Bias** | Markdown/structured responses score higher | Add "ignore formatting" to rubric, or strip formatting before judging |

These biases are well-documented in the literature (Zheng et al., 2023 "Judging LLM-as-a-Judge") and should always be considered when designing evaluation pipelines.

-

---

> **Note:** Parts 4-6 from the original notebook (DeepEval/RAGAS internals, customization) are now covered hands-on in Notebooks 03-05, where you'll see these concepts applied to real RAG pipeline results.

-

## Part 7: Best Practices and When NOT to Use LLM-as-a-Judge

### Best Practices

1. **Use the strongest available model as judge.** GPT-4o is a better judge than GPT-4o-mini, which is better than GPT-3.5. The judge should always be at least as capable as the model being evaluated.

2. **Design clear rubrics with examples.** Vague criteria like "rate quality" produce inconsistent results. Specific rubrics with score-level descriptions are essential.

3. **Run evaluations at temperature 0.** As demonstrated above, higher temperatures introduce unnecessary variance.

4. **Debias pairwise comparisons.** Always run both orderings and take consensus.

5. **Validate judge alignment with human scores.** Before deploying LLM-as-a-Judge, run a calibration study comparing LLM scores to human annotator scores on the same test set. Aim for high Spearman/Kendall correlation.

6. **Cache results to save cost.** The same (query, response, criteria) triple should always produce the same score (at temp=0). Cache aggressively.

7. **Use structured output.** JSON with mandatory reasoning fields prevents the judge from giving scores without justification.

8. **Be explicit about what NOT to evaluate.** Tell the judge "do not penalize brevity" or "ignore formatting" if those are not part of your criteria.

### When NOT to Use LLM-as-a-Judge

LLM-as-a-Judge is powerful but not always appropriate:

| Scenario | Better Alternative |
|---|---|
| **Exact match tasks** ("What is 2+2?") | Use exact string match or numeric comparison |
| **JSON/schema validation** ("Is the output valid JSON?") | Use `json.loads()` or a JSON schema validator |
| **Deterministic, reproducible scores needed** | Use rule-based metrics or embedding similarity |
| **Cost is prohibitive** | Use lighter metrics (BLEU, ROUGE, BERTScore) as proxies |
| **Judge is the same model being evaluated** | Self-enhancement bias will skew results |
| **Evaluating factual accuracy of cutting-edge knowledge** | The judge may not know the correct answer either |
| **High-stakes decisions (medical, legal)** | Use human experts — LLM judges can be confidently wrong |

### Cost Analysis

Let's estimate the cost of running LLM-as-a-Judge evaluations at scale.

In [ ]:
# Approximate token costs (as of early 2025, may change)
# GPT-4o: $2.50 per 1M input tokens, $10 per 1M output tokens
# GPT-4o-mini: $0.15 per 1M input tokens, $0.60 per 1M output tokens

pricing = {
    "gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
}

# Typical token counts per evaluation
avg_input_tokens = 800   # prompt + rubric + query + response
avg_output_tokens = 150  # score + reason

# DeepEval Faithfulness requires ~2-3 LLM calls per test case
# RAGAS Answer Relevancy requires ~2 LLM calls + 1 embedding call
avg_calls_per_metric = 2.5

def estimate_cost(n_test_cases, n_metrics, model, calls_per_metric=2.5):
    total_calls = n_test_cases * n_metrics * calls_per_metric
    input_cost = total_calls * avg_input_tokens * pricing[model]["input"]
    output_cost = total_calls * avg_output_tokens * pricing[model]["output"]
    return input_cost + output_cost

print("Cost Estimation for LLM-as-a-Judge")
print("=" * 60)
print(f"Assumptions: ~{avg_input_tokens} input tokens, ~{avg_output_tokens} output tokens per call")
print(f"             ~{avg_calls_per_metric} LLM calls per metric per test case\n")

scenarios = [
    (50, 3, "Small eval: 50 test cases, 3 metrics"),
    (200, 5, "Medium eval: 200 test cases, 5 metrics"),
    (1000, 5, "Large eval: 1000 test cases, 5 metrics"),
    (5000, 8, "Enterprise eval: 5000 test cases, 8 metrics"),
]

for n_cases, n_metrics, desc in scenarios:
    cost_4o = estimate_cost(n_cases, n_metrics, "gpt-4o")
    cost_mini = estimate_cost(n_cases, n_metrics, "gpt-4o-mini")
    print(f"{desc}:")
    print(f"  GPT-4o:      ${cost_4o:.2f}")
    print(f"  GPT-4o-mini: ${cost_mini:.2f}")
    print(f"  Savings:     {(1 - cost_mini/cost_4o)*100:.0f}% with mini\n")

The cost difference between GPT-4o and GPT-4o-mini is substantial at scale. A common pattern is to use GPT-4o-mini for rapid iteration during development and GPT-4o for final evaluation runs.

-

## Part 8: Summary

### The Three Paradigms

| Paradigm | Use When | Pros | Cons |
|---|---|---|---|
| **Pointwise** | You need to score individual responses | Simple, supports criteria-specific scoring | Absolute scores are hard to calibrate |
| **Pairwise** | You need to compare two systems | More reliable relative judgments | Suffers from position bias, does not scale well with many candidates |
| **Reference-Based** | You have gold-standard answers | Grounded in a known-good answer | Requires reference answers, penalizes valid alternatives |

### Biases and Mitigations

| Bias | Mitigation |
|---|---|
| Position bias | Run both orderings, take consensus |
| Verbosity bias | Explicit rubric: "do not favor length" |
| Self-enhancement bias | Use a different/stronger model as judge |
| Format bias | Strip formatting or add "ignore formatting" to rubric |

### DeepEval vs RAGAS

Both frameworks build on LLM-as-a-Judge but take different approaches:

- **DeepEval**: Direct LLM classification for most metrics, supports G-Eval for custom criteria, verbose mode for debugging
- **RAGAS**: Hybrid approach combining LLM generation with embedding similarity, more novel metric designs

### What You Now Know

You have built LLM-as-a-Judge **from scratch**: pointwise, pairwise, and reference-based judges. You have seen the biases that affect judge reliability and how to mitigate them. You have peeked inside DeepEval and RAGAS to understand that every metric they compute is ultimately a structured LLM-as-a-Judge pipeline. And you know when to use this approach and when simpler alternatives are more appropriate.

**This is the engine behind everything in this workbook series.** Every metric in notebooks 03 through 10 is, at its core, an LLM-as-a-Judge call wrapped in a framework.

-

**Next:** Continue to [01_environment_setup.ipynb](01_environment_setup.ipynb) to install all dependencies and configure your environment for the rest of the series.